<a href="https://colab.research.google.com/github/hossain8078/Machine-Learning-and-Deep-Learning/blob/main/Lab_7_Cloud_Deployment_Research_GCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Exercise 7: Researching Cloud Deployment Options

**Objective:** In this lab, you will research and compare different cloud services for deploying machine learning models. This is a research-based exercise designed to familiarize you with the cloud deployment landscape.

### Instructions

Choose one of the following cloud providers: **Amazon Web Services (AWS)**, **Google Cloud Platform (GCP)**, or **Microsoft Azure**.

For your chosen provider, research and write a short report (1-2 pages) that answers the following questions:

1.  **PaaS (Platform-as-a-Service) Options:**
    - Identify and describe a PaaS offering for deploying a web application (like a Flask API). Examples include AWS Elastic Beanstalk, Google App Engine, or Azure App Service.
    - What are the key features of this service?
    - What are the pros and cons of using this service for deploying a machine learning model?
2.  **CaaS (Containers-as-a-Service) Options:**
    - Identify and describe a CaaS offering for deploying Docker containers. Examples include AWS ECS, Google Cloud Run, or Azure Container Instances.
    - How does this service work?
    - What are the pros and cons of using this service compared to the PaaS option?
3.  **FaaS (Functions-as-a-Service) / Serverless Options:**
    - Identify and describe a serverless offering. Examples include AWS Lambda, Google Cloud Functions, or Azure Functions.
    - How would you deploy a machine learning model using this service?
    - What are the benefits and drawbacks of the serverless approach?
4.  **Recommendation:**
    - Based on your research, which service would you recommend for deploying a simple machine learning model for a small-scale application? Justify your choice.

---
# Report: Deploying ML Models on Google Cloud Platform (GCP)

**Chosen provider:** Google Cloud Platform

**Scenario used throughout:** the Iris `DecisionTreeClassifier` from the example notebook, saved as `iris_model.pkl` and served by the Flask `app.py` with a `/predict` endpoint.

| Model | GCP service | You manage | Google manages |
|---|---|---|---|
| PaaS | App Engine | Code + `app.yaml` | Servers, OS, runtime, scaling |
| CaaS | Cloud Run | Container image (`Dockerfile`) | Servers, scaling, HTTPS, load balancing |
| FaaS | Cloud Run functions (formerly Cloud Functions) | One function | Everything else |

> **📝 Note (Overview):** GCP gives three ways to host the same model. Going down the table, you manage less and Google manages more. The trade-off is control: App Engine gives you the least say over the environment, Cloud Run lets you define it fully through a container, and functions hide almost everything.

## 1. PaaS: Google App Engine

**What it is.** App Engine is GCP's fully managed platform for web apps. You upload source code plus a small config file, and Google provisions the servers, installs the Python runtime, routes traffic and scales instances. It has two environments: **Standard** (sandboxed runtimes, scales to zero, starts fast) and **Flexible** (runs in Docker containers on Compute Engine VMs, more customisable but always has at least one instance running).

**Key features**
- Supported language runtimes (Python, Java, Node.js, Go, PHP, Ruby) with no server setup.
- Automatic scaling on request load, including scale to zero in the Standard environment.
- **Versions and traffic splitting**: deploy a new model version and send, say, 10% of traffic to it (useful for A/B testing models).
- Built-in HTTPS, custom domains, Cloud Logging and Cloud Monitoring.
- Simple deployment: `gcloud app deploy`.

**Deploying the Flask model** needs `app.py`, `iris_model.pkl`, `requirements.txt` and:
```yaml
# app.yaml
runtime: python312
entrypoint: gunicorn -b :$PORT app:app
instance_class: F2
automatic_scaling:
  max_instances: 2
```

**Pros for ML models**
- Fastest path from a Flask notebook demo to a public URL; no Docker knowledge needed.
- Autoscaling, logging and HTTPS come out of the box.
- Traffic splitting supports safe rollout and rollback of model versions.

**Cons for ML models**
- Standard instances have limited memory and no GPUs, which rules out large deep-learning models.
- Restricted to supported runtimes; system libraries are hard to add in Standard.
- Only one App Engine app per project, and the platform is GCP-specific (vendor lock-in).
- Flexible environment cannot scale to zero, so it costs money even when idle.

> **📝 Note (Result of Section 1):** App Engine is the easiest way to put the Flask model online. You only need the code and a short `app.yaml`. It suits small, lightweight models like the Iris tree. Its limits (memory, no GPUs, fixed runtimes, GCP lock-in) make it a poor fit for large or unusual ML models.

## 2. CaaS: Google Cloud Run

**What it is.** Cloud Run is a fully managed, serverless platform that runs **any stateless Docker container** that listens for HTTP requests. It is built on the open-source Knative standard.

**How it works**
1. Write a `Dockerfile` that packages Python, the dependencies, `app.py` and `iris_model.pkl`.
2. Build the image and push it to **Artifact Registry** (or let `gcloud run deploy --source .` do both using Cloud Build).
3. Deploy the image as a Cloud Run **service**. Google gives it an HTTPS URL.
4. Each container instance handles many concurrent requests (default up to 80). Cloud Run adds instances as traffic grows and scales to zero when idle. Billing is per 100 ms of CPU and memory while requests are handled.
5. Each deploy creates an immutable **revision**, and traffic can be split between revisions.

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py iris_model.pkl ./
CMD exec gunicorn --bind :$PORT --workers 1 --threads 8 app:app
```
```bash
gcloud run deploy iris-api --source . --region us-central1 --allow-unauthenticated
```

**Pros compared with App Engine (PaaS)**
- **Any language, library or OS package**: the container defines the environment, so heavy ML dependencies (PyTorch, TensorFlow, system libraries) are not a problem.
- **Portability**: the same image runs on a laptop, on GKE, or on AWS/Azure container services, which reduces lock-in.
- Scales to zero, so a low-traffic model costs very little, with per-request billing.
- Larger resource limits (up to 32 GiB memory, and optional GPU support).
- Many services per project, versus one App Engine app.

**Cons compared with App Engine (PaaS)**
- You must learn Docker and maintain the image (base image updates, security patches).
- Cold starts when scaling from zero, made worse by large images or large models loaded at startup (can be avoided by setting `min-instances=1`, at extra cost).
- Containers are stateless; the model must be baked into the image or loaded from Cloud Storage.

> **📝 Note (Result of Section 2):** Cloud Run runs the same Flask app inside a Docker container. Compared with App Engine, it gives more freedom (any library, bigger models, portable image) and costs nothing when idle. The price is that you must write and maintain a `Dockerfile` and accept possible cold starts.

## 3. FaaS / Serverless: Cloud Run functions (formerly Google Cloud Functions)

**What it is.** A Functions-as-a-Service product: you write a single function, choose a trigger, and Google runs it on demand. In 2024 Google renamed Cloud Functions (2nd gen) to **Cloud Run functions**; it now runs on Cloud Run infrastructure, but the developer experience is still "write one function, no Dockerfile". Triggers include HTTP requests, Cloud Storage uploads, Pub/Sub messages and Cloud Scheduler.

**How to deploy the ML model**
1. Upload `iris_model.pkl` to a Cloud Storage bucket (keeps the code package small and lets you swap models).
2. Write a function that loads the model **once in global scope**, so warm invocations reuse it, then predicts on each request:
```python
# main.py
import functions_framework, joblib
from google.cloud import storage

storage.Client().bucket("my-models").blob("iris_model.pkl").download_to_filename("/tmp/model.pkl")
model = joblib.load("/tmp/model.pkl")
NAMES = ["setosa", "versicolor", "virginica"]

@functions_framework.http
def predict(request):
    features = request.get_json(force=True)["features"]
    return {"prediction": NAMES[int(model.predict([features])[0])]}
```
3. List `scikit-learn`, `joblib`, `google-cloud-storage` and `functions-framework` in `requirements.txt`, then deploy:
```bash
gcloud functions deploy iris-predict --gen2 --runtime python312 \
  --trigger-http --entry-point predict --region us-central1 --memory 512Mi
```
The same function could instead be triggered by a file upload to run **batch predictions** on new data.

**Benefits**
- No servers, no Dockerfile, no web framework: the least code and operations of all three options.
- Scales automatically from zero to many instances; you pay only while the function runs, and a generous monthly free tier covers small projects.
- Event triggers make it easy to automate ML workflows (score new files, retrain on a schedule).

**Drawbacks**
- **Cold starts**: importing scikit-learn and downloading the model adds latency to the first request after idle time.
- Memory, timeout and deployment-package limits make it unsuitable for large models or long inference jobs; no GPUs.
- Stateless and harder to debug locally and to monitor across many small functions.
- One function per endpoint becomes awkward when an API grows to many routes.

> **📝 Note (Result of Section 3):** A serverless function is the quickest and cheapest option for occasional or event-driven predictions, because you write one function and pay only per call. Cold starts, size and time limits, and harder debugging make it weaker for a busy, low-latency prediction API or a large model.

## 4. Recommendation

**For a simple ML model in a small-scale application, I recommend Google Cloud Run.**

**Justification**
1. **Cost**: it scales to zero and bills per request, so a small app with occasional traffic costs little or nothing within the free tier. App Engine Flexible would charge for an always-on VM.
2. **Reuses the existing work**: the Flask `app.py` from the example notebook runs unchanged inside a container. FaaS would mean rewriting it as a function.
3. **No dependency limits**: scikit-learn, NumPy and any future library (or a bigger model) fit in the container, unlike the more restricted App Engine Standard runtimes and the size limits of functions.
4. **Portability and growth**: the same Docker image can later move to GKE or another cloud, and revisions with traffic splitting support model versioning and A/B testing.
5. **Low operations**: like PaaS, Google handles servers, HTTPS and autoscaling. The only extra step is a short `Dockerfile`, and `gcloud run deploy --source .` can build it for you.

**When I would choose differently**
- **App Engine Standard** if the team has no Docker experience and just wants the simplest "push code" workflow.
- **Cloud Run functions** if predictions are event-driven (for example, scoring each new CSV uploaded to Cloud Storage) rather than a continuously used API.

**Production checklist (from the lecture):** require authentication (IAM or API keys) instead of `--allow-unauthenticated` for private models, monitor latency and errors with Cloud Monitoring, watch for data drift, and keep models versioned so a bad release can be rolled back.

*Note: service names, limits and free-tier amounts change over time; check the current GCP documentation before deploying.*

> **📝 Note (Result of Section 4):** Overall result: **Cloud Run** gives the best balance for a small ML app. It is about as easy as PaaS, about as cheap as FaaS at low traffic, and as flexible as containers, and it reuses the existing Flask code unchanged. App Engine or functions are better only in the specific cases listed above.